# ChatShop — Implementation Walkthrough

This notebook traces the entire data pipeline from raw JSON to a working hybrid RAG system. Run cells top to bottom.

**What we'll cover:**
1. The data model (`Product`)
2. Loading raw JSON
3. Cleaning & normalising records
4. Embedding text with sentence-transformers
5. Storing vectors + metadata in ChromaDB
6. Semantic search
7. Hybrid search — semantic + metadata filters
8. The full RAG chain (retrieval → prompt → LLM)

---
**Architecture at a glance:**
```
headphones.json
      │
      ▼
 load_json()          ← loader.py
      │
      ▼
 clean_headphones()   ← cleaner.py  →  Product(extra_attrs={wireless, anc, …})
      │
      ▼
 Embedder.encode()    ← embedder.py →  [0.12, -0.34, …]  (384-dim vector)
      │
      ▼
 ChromaStore.upsert() ← chroma.py   →  vector + flat metadata stored together
      │
  query time
      │
 ChromaStore.query(vector, where={price: {$lte:200}, wireless: True})
      │
      ▼
 Retriever            ← retriever.py
      │
      ▼
 RAGChain             ← chain.py     →  LLM answer
```

## 0 — Setup

Add `src/` to the Python path so we can import the `chatshop` package without installing it.

In [ ]:
import sys
from pathlib import Path

# Resolve the repo root regardless of where Jupyter was launched from
REPO_ROOT = Path("__file__").resolve().parent.parent
SRC = REPO_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Repo root:", REPO_ROOT)
print("src on path:", SRC.exists())

---
## 1 — The Data Model: `Product`

`Product` is a Pydantic model — the single source of truth that flows through every layer of the system.

### Key design decisions

| Field | Purpose |
|---|---|
| `product_id` | Stable key used as ChromaDB document ID (upsert is idempotent) |
| `title` + `description` | Only these two go into the **embedding**. Price and rating are filter concerns, not semantic ones. |
| `price`, `rating`, `rating_count` | Stored as **metadata** so ChromaDB can filter on them (`$lte`, `$gte`, etc.) |
| `extra_attrs` | Open dict for category-specific fields (e.g. `wireless`, `anc`, `battery_hours`). Flattened to scalars before writing to ChromaDB. |

The `extra_attrs` trick is what makes the schema future-proof: adding `television.json` just needs a new cleaner function — the model and the vectorstore don't change.

In [ ]:
from chatshop.data.models import Product

# Build a product by hand so we can inspect every method
p = Product(
    product_id="demo-001",
    title="Sony WH-1000XM5",
    description="Industry-leading ANC with 30-hour battery life.",
    category="headphones",
    price=349.99,
    rating=4.8,
    rating_count=12_500,
    extra_attrs={
        "brand": "Sony",
        "wireless": True,
        "anc": True,
        "battery_hours": 30,
        "type": "over-ear",
        "use_cases": "travel, office, studio",
    },
)

print("=== to_document_text() — what gets EMBEDDED ===")
print(p.to_document_text())
print()
print("=== to_metadata() — what gets stored as FILTERS ===")
import json
print(json.dumps(p.to_metadata(), indent=2))

Notice that `to_document_text()` contains **only** title and description. Price, rating, and boolean attributes are deliberately excluded from the embedding — they would add noise to the semantic space and are better handled as exact-match filters.

`to_metadata()` flattens `extra_attrs` into the top-level dict. ChromaDB metadata must be a `dict[str, scalar]` — no nested objects allowed, which is why lists (like `use_cases`) get joined to a comma-separated string during cleaning.

---
## 2 — Loading Raw Data

`load_json` is intentionally trivial — just `json.load`. No pandas, no schema validation yet. That's the cleaner's job.

In [ ]:
from chatshop.data.loader import load_json

JSON_PATH = REPO_ROOT / "data" / "headphones.json"

raw = load_json(JSON_PATH)

print(f"Loaded {len(raw)} records.\n")
print("First record (raw, unmodified):")
print(json.dumps(raw[0], indent=2))

The raw record has the schema as it came from the data source:
- `id` (int) — needs to become a string product ID
- `brand` + `name` — both needed to form a useful title
- `price_usd` — field name differs from our model's `price`
- `use_cases` is a **list** — needs to be flattened to a string for ChromaDB
- `waterproof_rating` can be `null`

All of this is the cleaner's responsibility.

---
## 3 — Cleaning: Raw Dict → `Product`

`clean_headphones` maps the headphones JSON schema to our canonical `Product` model. It handles:
- Field renaming (`price_usd` → `price`, `id` → `product_id`)
- Title construction (`brand + name`)
- Category-specific extras → `extra_attrs`
- List → comma-joined string for ChromaDB compatibility
- Deduplication by `product_id`
- Silently drops records with missing required fields

In [ ]:
from chatshop.data.cleaner import clean_headphones

products = clean_headphones(raw)

print(f"{len(raw)} raw records → {len(products)} clean Products\n")

# Show the first product and compare to the raw record above
p0 = products[0]
print(f"product_id : {p0.product_id}")
print(f"title      : {p0.title}")
print(f"price      : {p0.price}")
print(f"category   : {p0.category}")
print(f"extra_attrs: {json.dumps(p0.extra_attrs, indent=2)}")
print()
print("--- document text (goes into embedding) ---")
print(p0.to_document_text())

In [ ]:
# Scan the whole dataset — useful sanity check before ingesting
print("Prices:  ", sorted(p.price for p in products if p.price))
print("Wireless:", sum(1 for p in products if p.extra_attrs.get("wireless")))
print("ANC:     ", sum(1 for p in products if p.extra_attrs.get("anc")))
print("Brands:  ", sorted({p.extra_attrs.get("brand") for p in products if p.extra_attrs.get("brand")}))

---
## 4 — Embeddings

`Embedder` wraps `sentence-transformers`. The default model is **`all-MiniLM-L6-v2`** — a 22M-parameter model that produces 384-dimensional L2-normalized vectors. It's fast, runs locally, and is strong enough for product search.

We embed `to_document_text()` — title + description. The model never sees price or attributes.

In [ ]:
from chatshop.embeddings.embedder import Embedder

embedder = Embedder()  # downloads model on first run, ~90 MB

# Embed the document text for every product
texts = [p.to_document_text() for p in products]
vectors = embedder.encode(texts)

print(f"Number of vectors : {len(vectors)}")
print(f"Vector dimension  : {len(vectors[0])}")
print(f"First 8 dims of v0: {[round(x, 4) for x in vectors[0][:8]]}")

In [ ]:
import numpy as np

# Because vectors are L2-normalised, dot product == cosine similarity
# Let's see which products are most similar to "WH-1000XM5" in embedding space

query_vec = np.array(vectors[0])  # Sony WH-1000XM5
sims = [
    (p.title, float(np.dot(query_vec, np.array(v))))
    for p, v in zip(products, vectors)
]
sims.sort(key=lambda x: x[1], reverse=True)

print("Cosine similarity to 'Sony WH-1000XM5':")
for title, sim in sims:
    print(f"  {sim:.3f}  {title}")

Products with similar descriptions end up close in embedding space — that's what the vector search exploits. Notice that **price** doesn't influence these distances at all: a $30 pair of earbuds with the same ANC description could rank near a $350 premium pair. That's intentional — price filtering happens via the `where` clause, not the vector.

---
## 5 — ChromaDB: Storing Vectors + Metadata

`ChromaStore` is a thin wrapper around ChromaDB. Each document stored is:
- **ID** → `product_id` (string, used for upsert idempotency)
- **Embedding** → the 384-dim vector from the embedder
- **Document** → the raw text that was embedded (for inspection)
- **Metadata** → flat dict: title, price, category, wireless, anc, …

We use `PersistentClient` so the index survives across Python sessions.

In [ ]:
from chatshop.vectorstore.chroma import ChromaStore
from chatshop.config import settings

# Use a temporary collection so this notebook doesn't pollute the production store
store = ChromaStore(
    persist_dir=str(REPO_ROOT / "chroma_db_notebook"),
    collection_name="headphones_demo",
)

print(f"Collection before upsert: {store.count()} docs")

store.upsert(products, vectors)

print(f"Collection after upsert : {store.count()} docs")

In [ ]:
# Peek at the raw data Chrome has on disk
raw_peek = store._collection.peek(limit=2)

print("IDs:", raw_peek["ids"])
print()
print("Documents stored (i.e. to_document_text() output):")
for doc in raw_peek["documents"]:
    print(" ", doc[:120], "...")
print()
print("Metadata for first doc:")
print(json.dumps(raw_peek["metadatas"][0], indent=2))

The metadata dict is flat — scalars only. Notice how `extra_attrs` fields (`wireless`, `anc`, `battery_hours`, `brand`, etc.) are at the top level, right alongside `price` and `rating`. ChromaDB can filter on any of them.

---
## 6 — Semantic Search (no filters)

A plain vector query: embed the user's text, find nearest neighbours by cosine similarity.

In [ ]:
def show_results(results, label=""):
    if label:
        print(f"\n{'─'*55}")
        print(f" {label}")
        print(f"{'─'*55}")
    for i, p in enumerate(results, 1):
        price = f"${p.price:.0f}" if p.price else "N/A"
        wireless = "wireless" if p.extra_attrs.get("wireless") else "wired"
        anc = "ANC" if p.extra_attrs.get("anc") else "no ANC"
        print(f"[{i}] {p.title}")
        print(f"     {price} | {wireless} | {anc} | {p.extra_attrs.get('type', '?')}")


query = "comfortable over-ear headphones for long work sessions"
query_vector = embedder.encode_one(query)
results = store.query(query_vector, top_k=5)

show_results(results, f'Semantic only: "{query}"')

In [ ]:
# Different query — should surface different products
query2 = "waterproof earbuds for running and gym"
results2 = store.query(embedder.encode_one(query2), top_k=5)
show_results(results2, f'Semantic only: "{query2}"')

---
## 7 — Hybrid Search: Semantic + Metadata Filters

This is the core of the Cycle 2 architecture. The `where` parameter is passed directly to ChromaDB, which applies it as a **pre-filter** before scoring vectors. Only documents that satisfy the filter are candidates for the nearest-neighbour search.

ChromaDB filter operators: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, `$and`, `$or`.

In [ ]:
# Filter: wireless AND ANC
where_wireless_anc = {
    "$and": [
        {"wireless": {"$eq": True}},
        {"anc": {"$eq": True}},
    ]
}

query = "noise cancelling headphones"
results = store.query(embedder.encode_one(query), top_k=5, where=where_wireless_anc)
show_results(results, f'"{query}" + wireless=True + anc=True')

In [ ]:
# Filter: price <= 100, wireless
where_budget = {
    "$and": [
        {"price": {"$lte": 100}},
        {"price": {"$gt": 0}},      # exclude sentinel -1.0 for products with no price
        {"wireless": {"$eq": True}},
    ]
}

query = "wireless headphones on a budget"
results = store.query(embedder.encode_one(query), top_k=5, where=where_budget)
show_results(results, f'"{query}" + price≤$100 + wireless')

In [ ]:
# Filter: specific brand
where_brand = {"brand": {"$eq": "Sony"}}

query = "premium sound quality"
results = store.query(embedder.encode_one(query), top_k=5, where=where_brand)
show_results(results, f'"{query}" + brand=Sony')

### Why hybrid beats semantic-only

Without a price filter, "budget wireless headphones" might return a $350 Sony because its description matches the word "wireless". With `price: {$lte: 100}` applied at the DB level, the expensive ones are excluded before the vector search even runs — so the top results are semantically relevant *and* affordable.

This approach is **one round-trip**: ChromaDB handles filtering and nearest-neighbour search in a single call. There's no separate SQL query, no post-processing filter.

---
## 8 — The Full RAG Chain

### How the layers stack up

```
User query
   │
   ▼
Retriever.retrieve(query)          ← embeds query → calls store.query()
   │   returns list[Product]
   ▼
build_user_message(query, products) ← formats products into a catalog block
   │   returns a prompt string
   ▼
litellm.completion(messages=[…])   ← sends system + user message to the LLM
   │
   ▼
LLM answer
```

### Step 8a: Retriever

In [ ]:
from chatshop.rag.retriever import Retriever

# Wire the retriever to the notebook's demo store (not the production one)
retriever = Retriever(embedder=embedder, store=store)

query = "wireless headphones with great noise cancellation"
retrieved = retriever.retrieve(query, top_k=3)

print(f"Query: '{query}'")
print(f"Retrieved {len(retrieved)} products:\n")
for p in retrieved:
    print(f"  • {p.title} — ${p.price}")

### Step 8b: Prompt Construction

Before sending anything to an LLM, let's see exactly what the prompt looks like. This is useful for debugging retrieval quality.

In [ ]:
from chatshop.rag.prompt import SYSTEM_PROMPT, build_user_message

user_message = build_user_message(query, retrieved)

print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print()
print("=== USER MESSAGE (query + retrieved catalog) ===")
print(user_message)

The retrieved products are formatted as a numbered catalog block, injected before the user's actual request. The LLM is instructed to only recommend products from this catalog — that's the "grounding" that prevents hallucination.

### Step 8c: LLM call (requires API key)

The cell below calls the LLM. Skip it if you don't have `LITELLM_API_KEY` set in `.env`.

In [ ]:
from chatshop.config import settings

if not settings.litellm_api_key:
    print("No LITELLM_API_KEY found in .env — skipping LLM call.")
    print(f"Model configured: {settings.litellm_model}")
else:
    from chatshop.rag.chain import RAGChain

    chain = RAGChain(retriever=retriever)
    query = "I need wireless headphones under $200 that are good for commuting"

    print(f"Query: {query}\n")
    print("=" * 55)
    answer = chain.run(query)
    print(answer)

---
## 9 — End-to-End: What the Production Ingestion Script Does

The notebook ran each step manually. In production, `scripts/ingest.py` wraps steps 2–5 in a single CLI call:

```bash
uv run python scripts/ingest.py --json data/headphones.json --category headphones
```

Internally that resolves to:
1. `load_json(path)` → raw list
2. `_JSON_CLEANERS["headphones"](raw)` → `list[Product]`
3. `embedder.encode(texts)` → vectors (in sub-batches to bound memory)
4. `store.upsert(products, vectors)` → written to ChromaDB in batches of 512

Let's verify the notebook's demo store matches what the script would produce.

In [ ]:
print(f"Demo store holds {store.count()} documents.")
print(f"That matches len(products) = {len(products)}: {store.count() == len(products)}")

# Upsert is idempotent — running again doesn't duplicate
store.upsert(products, vectors)
print(f"After second upsert: {store.count()} documents (unchanged).")

---
## 10 — Adding a New Category (extensibility demo)

The `extra_attrs` design means adding, say, `television.json` requires only:
1. A new cleaner function `clean_televisions()` in `cleaner.py`
2. Registering it in `_JSON_CLEANERS` in `ingest.py`

No changes to `Product`, `ChromaStore`, `Embedder`, or `Retriever`.

Here's what that cleaner would look like — we can stub it right here to prove the point:

In [ ]:
# Hypothetical television record
tv_record = {
    "id": "tv-001",
    "brand": "Samsung",
    "name": "QN90C Neo QLED 65\"",
    "price_usd": 1799.99,
    "description": "Quantum Matrix Technology with Mini LEDs delivers precise brightness control.",
    "screen_size_inches": 65,
    "resolution": "4K",
    "smart_tv": True,
    "hdmi_ports": 4,
    "refresh_rate_hz": 120,
}

# The same Product model, different extra_attrs
tv = Product(
    product_id=str(tv_record["id"]),
    title=f"{tv_record['brand']} {tv_record['name']}",
    description=tv_record["description"],
    category="television",
    price=tv_record["price_usd"],
    extra_attrs={
        "brand": tv_record["brand"],
        "screen_size_inches": tv_record["screen_size_inches"],
        "resolution": tv_record["resolution"],
        "smart_tv": tv_record["smart_tv"],
        "hdmi_ports": tv_record["hdmi_ports"],
        "refresh_rate_hz": tv_record["refresh_rate_hz"],
    },
)

print("TV metadata (filterable fields):")
print(json.dumps(tv.to_metadata(), indent=2))

The TV's category-specific fields (`screen_size_inches`, `hdmi_ports`, `refresh_rate_hz`) live in `extra_attrs` but land as top-level metadata keys in ChromaDB — queryable with `where={"screen_size_inches": {"$gte": 55}, "smart_tv": True}`. The rest of the system is completely unaware.

---
## Cleanup

Delete the demo ChromaDB collection created by this notebook (the production collection is untouched).

In [ ]:
import shutil

demo_db_path = REPO_ROOT / "chroma_db_notebook"
if demo_db_path.exists():
    shutil.rmtree(demo_db_path)
    print(f"Deleted {demo_db_path}")
else:
    print("Nothing to clean up.")